# 32 — Trade Intelligence Analysis (Problem 3) — KPI (LOCKED)

**Business question:** How dependent is each country on imports/exports?

**Scope:** 2001–2020, `marts` schema.

**Status: LOCKED — re-run confirmed the fix.** A second, separate bug was found after this
notebook was originally locked: the aggregate-exclusion logic described below (a join
against `fact_socioeconomic__population`'s `DISTINCT area_code`) does **not** actually
exclude regional/income-group aggregates — those composites have population rows too
(aggregates are sums), so they passed the join undetected. Worse, KPI 1's reporter side
and KPI 2/3's reporter side had **no aggregate filter applied at all** — only KPI 2/3's
partner side was filtered, and via the same broken method. A live diagnostic found
4,400 aggregate rows (area_code >= 5000) in `fact_trade__trade` alone within
2001–2020. **Fixed** by filtering both the reporter and partner sides of every KPI
query directly to `area_code < 5000` (or `reporter_country_code`/
`partner_country_code < 5000` for `fact_trade__matrix`, which stores these as BIGINT)
— the confirmed universal country-vs-aggregate boundary across all 4 source tables in
this project. KPI 4 needed no direct fix since it's built from KPI 1's now-cleaned
output. **Confirmed real counts after re-run:** KPI 1: 4,414 rows (down from 4,934
pre-fix), KPI 2: 2,713 rows, KPI 3: 2,870 rows, KPI 4: 216 countries with both 2001 and
2020 data. The 9-unmatched-partner figure quoted below predates this fix and is no
longer the relevant check (aggregate exclusion is now a direct `area_code < 5000`
filter, not a population-table match). Everything else in this
notebook (item selection, KPI redefinition, element-casing fixes) remains valid and is
preserved below. Two separate representative items are used, by necessity: `31_`'s
item-listing and cross-reference cells were run for real, plus a direct
shell query against `fact_trade__matrix`:

- **`TEST_ITEM = "Pesticides (total)"`** — used for KPI 1 (`fact_trade__trade` +
  `fact_trade__matrix`-independent). `fact_trade__trade`'s 37 items are all
  pesticides/agrochemicals; this is the best-covered one (21,736 rows, 280 countries,
  full 2001–2020).
- **`MATRIX_ITEM = "Food preparations n.e.c."`** — used for KPI 2 and KPI 3 (both built
  on `fact_trade__matrix`). **Confirmed, not a naming mismatch:** none of
  `fact_trade__trade`'s 37 pesticide items exist in `fact_trade__matrix` under any
  matching name — a direct `LIKE '%pestic%'/'%insecticid%'/'%herbicid%'/'%fungicid%'`
  search returned 0 rows. `fact_trade__matrix`'s item universe is entirely processed
  food/beverage/agricultural commodities, with no pesticide coverage at all. This is the
  best-covered item there (595,831 rows, 186 reporters, full 2001–2020).

**Consequence — KPI 1 uses a different commodity than KPI 2/3.** This notebook now
answers two related but distinct questions: how import-dependent are countries on
pesticides (KPI 1, KPI 4), and how concentrated/diversified are countries' trade
partners for a food-preparation commodity (KPI 2, KPI 3). If these feed the same Power
BI view, the item difference should be labeled clearly rather than implied as one
consistent commodity story.

**KPI 1 also redefined:** the original plan was
`imports ÷ (production + imports − exports)`, using `fact_production__crops_livestock`'s
`Production` element as the domestic-supply denominator. That table has no "production"
figure for a pesticide, so KPI 1 is now a pure trade-based ratio instead:
`imports ÷ (imports + exports)`, using `fact_trade__trade` alone — see Section 3.

**Other fixes applied, code-level:**
- Element strings corrected to the real casing: `"Import value"`, `"Import quantity"`,
  `"Export value"`, `"Export quantity"`.
- Aggregate-exclusion join applied to `fact_trade__trade` (KPI 1) and to
  `fact_trade__matrix`'s partner side (KPI 2/3) — `31_`'s partner-side check found 9
  unmatched partner codes (uninhabited/remote territories, not regional aggregates).
- KPI 2's top-partner selection logic was rewritten (the previous `.sort()` + `.first()`
  pattern isn't reliable in Polars).
- Guard cells raise immediately if either item has no coverage in a needed table,
  instead of producing silently-empty KPI tables (this is exactly how the
  `MATRIX_ITEM` mismatch was caught).

**KPIs:**
1. Import Dependency Ratio (pesticides) — imports ÷ (imports + exports), per
   country-year *(redefined from the original production-based formula — Section 3)*
2. Export Concentration (food preparations, top-partner share) — largest partner's
   share of a country's export value, per country-year, from `fact_trade__matrix`
3. Import Partner Diversification (food preparations) — count of partners supplying at
   least `SIGNIFICANT_PARTNER_SHARE_PCT` of a country's imports, per country-year
4. Trade Dependency Trend (pesticides, 2001 → 2020 change, ranked) — same two-point
   comparison pattern used in `12_` KPI 3 and `22_` KPI 4, applied to KPI 1

## 1. Setup

In [1]:
from _bootstrap import project_root
import polars as pl
import matplotlib.pyplot as plt

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(120)
pl.Config.set_tbl_rows(50)

from src.database.connection import get_duckdb_conn

conn = get_duckdb_conn(read_only=True)
print("Connected")

Connected


In [2]:
SCHEMA = "marts"
YEAR_START, YEAR_END = 2001, 2020

# CONFIRMED from a real run of 31_'s item-listing cell against the live DB:
# fact_trade__trade's 37 items are ALL pesticides/agrochemicals, not crops or food
# commodities. "Pesticides (total)" is the best-covered item (21,736 rows, 280
# countries, full 2001-2020 range) and is used as TEST_ITEM for KPI 1 below.
TEST_ITEM = "Pesticides (total)"

# CONFIRMED via a direct shell query against fact_trade__matrix: none of
# fact_trade__trade's 37 pesticide items exist there under any matching name (a
# LIKE '%pestic%'/'%insecticid%'/'%herbicid%'/'%fungicid%' search returned 0 rows).
# fact_trade__matrix's item universe is entirely processed food/beverage/agricultural
# commodities (e.g. "Food preparations n.e.c.", "Wine", "Cheese from whole cow milk") —
# no overlap with fact_trade__trade's pesticide-only scope at all, confirmed, not a
# naming mismatch. KPI 2 and KPI 3 (both built on fact_trade__matrix) therefore use a
# SEPARATE item from TEST_ITEM: MATRIX_ITEM = "Food preparations n.e.c.", the
# best-covered item in fact_trade__matrix (595,831 rows, 186 reporters, 2001-2020).
# KPI 1 (pesticide import dependency) and KPI 2/3 (food-preparation export
# concentration / import diversification) are therefore about two different
# commodities, not one shared story -- flagged again in Section 4/5 below and in the
# Findings section.
MATRIX_ITEM = "Food preparations n.e.c."

# Minimum import share (%) for a partner to count as "significant" in KPI 3's
# diversification count. Placeholder threshold — revisit once the real distribution of
# partner shares is visible (e.g. via a histogram) rather than picking blind.
SIGNIFICANT_PARTNER_SHARE_PCT = 5.0


## 2. Confirm source indicators exist with expected coverage

One check, not open-ended exploration — confirms the exact item/element strings still
match `marts` and cover 2001–2020 before building KPIs on top of them.

**Before finalizing this notebook:** replace the element strings below
(`'Import Quantity'`, `'Export Quantity'`, `'Import Value'`, `'Export Value'`) with
whatever `31_` Section 6 actually returns for `fact_trade__trade` and
`fact_trade__matrix` — these are carried over as placeholders matching common FAOSTAT
naming conventions, not confirmed against this project's `marts` build.

In [3]:
check_trade = conn.execute(f"""
    SELECT item, element, COUNT(*) AS n_rows, COUNT(DISTINCT area_code) AS n_countries,
           MIN(year) AS year_min, MAX(year) AS year_max
    FROM {SCHEMA}.fact_trade__trade
    WHERE item = ?
      AND year BETWEEN {YEAR_START} AND {YEAR_END}
    GROUP BY item, element
    ORDER BY element
""", [TEST_ITEM]).pl()

check_trade

item,element,n_rows,n_countries,year_min,year_max
str,str,i64,i64,i64,i64
"""Pesticides (total)""","""Export quantity""",5354,272,2001,2020
"""Pesticides (total)""","""Export value""",5354,272,2001,2020
"""Pesticides (total)""","""Import quantity""",5514,280,2001,2020
"""Pesticides (total)""","""Import value""",5514,280,2001,2020


In [4]:
# Uses MATRIX_ITEM, not TEST_ITEM -- fact_trade__matrix has no pesticide items at all
# (confirmed via direct shell query), so this check is against the separate
# food-preparation item used for KPI 2/3. See Cell 3's comment for full rationale.
check_matrix = conn.execute(f"""
    SELECT item, element, COUNT(*) AS n_rows,
           COUNT(DISTINCT reporter_country_code) AS n_reporters,
           COUNT(DISTINCT partner_country_code) AS n_partners,
           MIN(year) AS year_min, MAX(year) AS year_max
    FROM {SCHEMA}.fact_trade__matrix
    WHERE item = ?
      AND year BETWEEN {YEAR_START} AND {YEAR_END}
    GROUP BY item, element
    ORDER BY element
""", [MATRIX_ITEM]).pl()

check_matrix

item,element,n_rows,n_reporters,n_partners,year_min,year_max
str,str,i64,i64,i64,i32,i32
"""Food preparations n.e.c.""","""Export quantity""",146167,182,200,2001,2020
"""Food preparations n.e.c.""","""Export value""",146167,182,200,2001,2020
"""Food preparations n.e.c.""","""Import quantity""",151747,186,200,2001,2020
"""Food preparations n.e.c.""","""Import value""",151750,186,200,2001,2020


In [5]:
# NOTE: TEST_ITEM = "Pesticides (total)" is a pesticide, not a crop -- this check
# against fact_production__crops_livestock's "Production" element is expected to
# return 0 rows (crop/livestock output has no "Pesticides (total)" entry) and is kept
# here only for visibility, not as a KPI 1 dependency anymore. KPI 1 below no longer
# uses this table (see Section 3 for the redefined, trade-only ratio).
check_production = conn.execute(f"""
    SELECT item, element, COUNT(*) AS n_rows, COUNT(DISTINCT area_code) AS n_countries,
           MIN(year) AS year_min, MAX(year) AS year_max
    FROM {SCHEMA}.fact_production__crops_livestock
    WHERE item = ?
      AND element = 'Production'
      AND year BETWEEN {YEAR_START} AND {YEAR_END}
    GROUP BY item, element
""", [TEST_ITEM]).pl()

check_production

item,element,n_rows,n_countries,year_min,year_max
str,str,i64,i64,i64,i64


In [6]:
# Guard: stop here (instead of silently producing near-empty KPIs below) if either
# item doesn't actually have usable coverage in the tables that matter.
if check_trade.height == 0:
    raise ValueError(
        f"TEST_ITEM='{TEST_ITEM}' returned 0 rows from fact_trade__trade -- "
        f"KPI 1 will be empty. Pick a different item from 31_'s item-listing/"
        f"cross-reference cells."
    )
if check_matrix.height == 0:
    raise ValueError(
        f"MATRIX_ITEM='{MATRIX_ITEM}' returned 0 rows from fact_trade__matrix -- "
        f"KPI 2 and KPI 3 will be empty. Pick a different item -- run the item-coverage "
        f"query used to choose 'Food preparations n.e.c.' again if this item's coverage "
        f"has changed in the marts build."
    )
# check_production is expected to be 0 rows for TEST_ITEM="Pesticides (total)" (a
# pesticide, not a crop) -- no longer a KPI 1 dependency, so not guarded as an error
# or warning here. See Section 3: KPI 1 was redefined to not need this table.

**Note:** if any check above returns fewer rows than expected, or zero rows for an
element string, that string no longer matches `marts` exactly (e.g. wording changed in a
rebuild, or the placeholder guess was wrong) — that's the only thing to investigate
before trusting the KPI queries below. This mirrors the same check pattern used in `22_`
Section 2.

## 3. KPI 1 — Import Dependency Ratio

**Definition (redefined for this item):** originally planned as import quantity ÷
apparent domestic supply (production + imports − exports). That formula assumed a food
crop, where `fact_production__crops_livestock`'s `Production` element is a meaningful
denominator. **`TEST_ITEM = "Pesticides (total)"` is not a crop — there is no
crop/livestock "production" of a pesticide** — so that denominator doesn't apply here.

**Redefined ratio, trade-only, no production table needed:**
`import_dependency_pct = imports / (imports + exports) * 100`, per country-year. This
answers a slightly narrower but still valid question: of this country's total trade
flow (imports + exports) in this item, what share is imports? A country at 100% only
imports and never exports; a country near 0% is a net exporter.

**Power BI use:** primary signal for "how import-dependent is this country" for the
selected item, given the item universe available in `fact_trade__trade` is
pesticides/agrochemicals, not food crops.

In [7]:
# Element casing corrected per 31_ Section 6 findings: 'Import quantity' / 'Export
# quantity', not Title Case. Aggregate-exclusion join applied per 31_ Section 5 finding —
# fact_trade__trade has 29 regional/aggregate rows (e.g. "Africa (excluding
# intra-trade)") that must be excluded before country-level ranking.
#
# REDEFINED (see Section 3 markdown): no join to fact_production__crops_livestock —
# TEST_ITEM is a pesticide, and that table's "Production" element is crop/livestock
# output, not applicable here. Ratio is now import quantity / (import + export
# quantity), using fact_trade__trade alone.
trade_qty_raw = conn.execute(f"""
    SELECT area_code, area, year, element, value
    FROM {SCHEMA}.fact_trade__trade
    WHERE item = ?
      AND element IN ('Import quantity', 'Export quantity')
      AND year BETWEEN {YEAR_START} AND {YEAR_END}
""", [TEST_ITEM]).pl().with_columns(pl.col("value").cast(pl.Float64, strict=False))

# Aggregate-exclusion FIXED: the previous "keep only areas that also appear in
# fact_socioeconomic__population" approach did NOT actually exclude aggregates --
# regional/income-group composites (World, Africa, SIDS, "Africa (excluding
# intra-trade)", etc.) have population rows too, so they passed this check.
# Confirmed via live diagnostic: fact_trade__trade has 4,400 rows with area_code >= 5000
# within 2001-2020, none of which were being excluded by the population-join proxy.
# The real, universal rule (confirmed across all 4 source tables in this project) is
# that every genuine country/territory area_code is < 5000 -- including the 5-digit
# "excluding intra-trade" variant codes (e.g. 51000), which are also numerically >= 5000.
trade_qty = trade_qty_raw.filter(pl.col("area_code").cast(pl.Utf8).str.strip_chars().cast(pl.Int64, strict=False) < 5000)

trade_pivot = trade_qty.pivot(values="value", index=["area_code", "area", "year"], on="element")

kpi1_dependency = (
    trade_pivot
    .with_columns(
        (pl.col("Import quantity").fill_null(0)
         + pl.col("Export quantity").fill_null(0)).alias("total_trade_qty")
    )
    .filter(pl.col("total_trade_qty") > 0)
    .with_columns(
        (pl.col("Import quantity").fill_null(0) / pl.col("total_trade_qty") * 100)
        .alias("import_dependency_pct")
    )
)

print(f"KPI 1 rows: {kpi1_dependency.height}")
kpi1_dependency.select(["area_code", "area", "year", "import_dependency_pct"]).head(10)

KPI 1 rows: 4414


area_code,area,year,import_dependency_pct
str,str,i64,f64
"""3""","""Albania""",2001,91.359205
"""3""","""Albania""",2002,80.469443
"""3""","""Albania""",2003,99.749971
"""3""","""Albania""",2004,99.674734
"""3""","""Albania""",2005,99.608666
"""3""","""Albania""",2006,99.117279
"""3""","""Albania""",2007,98.973185
"""3""","""Albania""",2008,99.519127
"""3""","""Albania""",2009,99.450146


**Caveat, confirmed in `31_` and refined above:** `fact_trade__trade`'s 37 items are
all pesticides/agrochemicals — `TEST_ITEM = "Pesticides (total)"` has no matching
"Production" record in `fact_production__crops_livestock`, so KPI 1 no longer joins to
that table (Section 3 redefinition). The ratio here reflects trade balance (imports
as a share of total trade flow), not import share of total domestic supply. If a
future representative item needs the original production-based formula, pick one that
exists in both `fact_trade__trade` and `fact_production__crops_livestock` (check `31_`'s
item cross-reference cell — `item_overlap`) and adapt Section 3's formula back to
`imports / (production + imports − exports)`.

## 4. KPI 2 — Export Concentration (top-partner share)

**Definition:** for each country-year, the largest single partner's share of that
country's total export value for `MATRIX_ITEM`, using `fact_trade__matrix`'s bilateral
detail. This is the piece `fact_trade__trade`'s country-level totals can't answer — it
speaks directly to the "diversify export markets" decision.

**Item note:** uses `MATRIX_ITEM = "Food preparations n.e.c."`, not `TEST_ITEM`.
Confirmed via direct query: `fact_trade__matrix` has no pesticide-related items at
all — its item universe is entirely processed food/beverage/agricultural commodities.
So KPI 1 (pesticide import dependency) and KPI 2/3 (food-preparation export
concentration / import diversification) describe two different commodities, not one
shared story — call this out explicitly if these KPIs go into the same Power BI view.

**Power BI use:** flags countries whose exports are concentrated in one buyer — a
country at or near 100% is fully dependent on a single export market.

In [8]:
# Aggregate-exclusion FIXED on BOTH sides. Two problems with the previous approach:
# (1) it only filtered the partner side, leaving the reporter side (reporter_country_code)
# completely unfiltered -- a regional aggregate could appear as a "country" in the
# ranking output; (2) the population-join proxy (valid_partner_areas) doesn't actually
# exclude aggregates, since composites have population rows too. Fixed by filtering
# both reporter_country_code and partner_country_code directly to < 5000, the confirmed
# universal country-vs-aggregate boundary. fact_trade__matrix's codes are BIGINT (unlike
# fact_trade__trade's VARCHAR), so no cast is needed here.
export_matrix_raw = conn.execute(f"""
    SELECT reporter_country_code, reporter_country_name AS area, year,
           partner_country_code, partner_country_name, value
    FROM {SCHEMA}.fact_trade__matrix
    WHERE item = ?
      AND element = 'Export value'
      AND year BETWEEN {YEAR_START} AND {YEAR_END}
      AND reporter_country_code < 5000
      AND partner_country_code < 5000
""", [MATRIX_ITEM]).pl()

export_matrix = export_matrix_raw

country_year_totals = (
    export_matrix
    .group_by(["reporter_country_code", "area", "year"])
    .agg(pl.col("value").sum().alias("total_export_value"))
)

# Top partner per country-year: use sort + group_by().head(1) via row-level ranking
# rather than .first() after a global sort, which is not guaranteed to preserve
# within-group order in Polars. pl.col(...).top_k(1) inside agg is the robust form.
top_partner = (
    export_matrix
    .sort(["reporter_country_code", "area", "year", "value"], descending=[False, False, False, True])
    .group_by(["reporter_country_code", "area", "year"], maintain_order=True)
    .agg(
        pl.col("value").max().alias("top_partner_value"),
        pl.col("partner_country_name").sort_by(pl.col("value"), descending=True).first().alias("top_partner"),
    )
)

kpi2_export_concentration = (
    country_year_totals
    .join(top_partner, on=["reporter_country_code", "area", "year"])
    .filter(pl.col("total_export_value") > 0)
    .with_columns(
        (pl.col("top_partner_value") / pl.col("total_export_value") * 100)
        .alias("top_partner_share_pct")
    )
)

# Note: aggregate exclusion now happens in the SQL WHERE clause above (both
# reporter_country_code and partner_country_code < 5000), not as a post-hoc Polars
# filter, so there's no separate "excluded" row count to report here anymore.
print(f"KPI 2 rows: {kpi2_export_concentration.height}")
kpi2_export_concentration.select(
    ["area", "year", "top_partner", "top_partner_share_pct"]
).sort("top_partner_share_pct", descending=True).head(10)

KPI 2 rows: 2713


area,year,top_partner,top_partner_share_pct
str,i32,str,f64
"""Albania""",2006,"""Serbia""",100.0
"""Albania""",2014,"""Serbia""",100.0
"""Albania""",2017,"""Serbia""",100.0
"""Albania""",2018,"""Serbia""",100.0
"""Albania""",2019,"""North Macedonia""",100.0
"""Albania""",2020,"""United States of America""",100.0
"""Bahamas""",2020,"""United States of America""",100.0
"""Solomon Islands""",2017,"""Nauru""",100.0
"""Solomon Islands""",2018,"""Nauru""",100.0


## 5. KPI 3 — Import Partner Diversification (count of significant partners)

**Definition:** count of partners each supplying at least `SIGNIFICANT_PARTNER_SHARE_PCT`
(currently a placeholder 5%) of a country's total import value for `MATRIX_ITEM`, per
country-year. A country sourcing from many "significant" partners is more diversified
(and presumably less exposed to a single-supplier shock) than one sourcing almost
entirely from a single partner even if that partner isn't literally its only supplier.

**Item note:** uses `MATRIX_ITEM`, the same food-preparation item as KPI 2 — see Cell 3
and KPI 2's markdown for why this differs from `TEST_ITEM`.

**Threshold caveat:** the 5% cutoff is not yet validated against the real distribution of
partner shares — revisit once this notebook has been run against `marts` and the actual
spread of `top_partner_share_pct`-style values is visible.

In [9]:
# Aggregate-exclusion FIXED on BOTH sides -- same rationale and same fix as KPI 2
# above: the previous approach only filtered the partner side, and did so via a
# population-join proxy that doesn't actually exclude aggregates. Without a reporter-
# side filter, a regional aggregate (e.g. "Africa") could appear as a "country" row in
# this ranking; without a correct partner-side filter, a regional aggregate could count
# as a "significant partner" and inflate a real country's diversification score with a
# non-country entity. Fixed by filtering both codes directly to < 5000.
import_matrix_raw = conn.execute(f"""
    SELECT reporter_country_code, reporter_country_name AS area, year,
           partner_country_code, partner_country_name, value
    FROM {SCHEMA}.fact_trade__matrix
    WHERE item = ?
      AND element = 'Import value'
      AND year BETWEEN {YEAR_START} AND {YEAR_END}
      AND reporter_country_code < 5000
      AND partner_country_code < 5000
""", [MATRIX_ITEM]).pl()

import_matrix = import_matrix_raw

import_totals = (
    import_matrix
    .group_by(["reporter_country_code", "area", "year"])
    .agg(pl.col("value").sum().alias("total_import_value"))
)

import_with_share = (
    import_matrix
    .join(import_totals, on=["reporter_country_code", "area", "year"])
    .filter(pl.col("total_import_value") > 0)
    .with_columns(
        (pl.col("value") / pl.col("total_import_value") * 100).alias("partner_share_pct")
    )
)

kpi3_diversification = (
    import_with_share
    .filter(pl.col("partner_share_pct") >= SIGNIFICANT_PARTNER_SHARE_PCT)
    .group_by(["reporter_country_code", "area", "year"])
    .agg(pl.col("partner_country_name").n_unique().alias("n_significant_partners"))
)

# Note: aggregate exclusion now happens in the SQL WHERE clause above (both
# reporter_country_code and partner_country_code < 5000), not as a post-hoc Polars
# filter, so there's no separate "excluded" row count to report here anymore.
print(f"KPI 3 rows: {kpi3_diversification.height}")
kpi3_diversification.sort("n_significant_partners").head(10)

KPI 3 rows: 2870


reporter_country_code,area,year,n_significant_partners
i64,str,i32,u32
138,"""Mexico""",2012,1
109,"""Jamaica""",2007,1
122,"""Lesotho""",2018,1
33,"""Canada""",2009,1
12,"""Bahamas""",2009,1
12,"""Bahamas""",2015,1
33,"""Canada""",2019,1
138,"""Mexico""",2020,1
209,"""Eswatini""",2002,1


## 6. KPI 4 — Trade Dependency Trend (2001 → 2020 change, ranked)

**Definition:** change in `import_dependency_pct` (KPI 1) from 2001 to 2020 per country,
for the selected item. Same two-point comparison pattern used for Food Security's KPI 3
(`12_`) and Productivity's KPI 4 (`22_`), applied here to the dependency ratio.

In [10]:
first_year = kpi1_dependency.filter(pl.col("year") == YEAR_START).select(
    ["area_code", "area", "import_dependency_pct"]
).rename({"import_dependency_pct": "dependency_2001"})

last_year = kpi1_dependency.filter(pl.col("year") == YEAR_END).select(
    ["area_code", "area", "import_dependency_pct"]
).rename({"import_dependency_pct": "dependency_2020"})

kpi4_trend = (
    first_year
    .join(last_year, on=["area_code", "area"], how="inner")
    .with_columns(
        (pl.col("dependency_2020") - pl.col("dependency_2001")).alias("dependency_change_pct_pts")
    )
    .sort("dependency_change_pct_pts", descending=True)
)

# Visibility into join drop-off: an inner join silently drops any country present in
# only one of the two years. Print counts so a small kpi4_trend isn't mistaken for a
# bug when it's really just sparse 2001 or 2020 coverage for this item.
print(f"Countries with {YEAR_START} data: {first_year.height}, "
      f"countries with {YEAR_END} data: {last_year.height}, "
      f"countries with both (KPI 4 rows): {kpi4_trend.height}")
kpi4_trend.head(10)

Countries with 2001 data: 219, countries with 2020 data: 222, countries with both (KPI 4 rows): 216


area_code,area,dependency_2001,dependency_2020,dependency_change_pct_pts
str,str,f64,f64,f64
"""55""","""Dominica""",16.25827,99.977717,83.719448
"""158""","""Niger""",31.438583,95.353416,63.914833
"""236""","""Venezuela (Bolivarian Republic of)""",47.585261,95.024994,47.439733
"""5""","""American Samoa""",53.903542,99.099073,45.195532
"""205""","""Western Sahara""",30.247793,70.28458,40.036787
"""101""","""Indonesia""",23.80794,63.82579,40.017851
"""211""","""Switzerland""",33.71288,65.192145,31.479264
"""63""","""Estonia""",23.704031,54.733506,31.029475
"""202""","""South Africa""",31.156221,59.71874,28.562519


## 7. Findings — status

**LOCKED — re-run confirmed the fix.** A later-discovered aggregate-exclusion bug (the
population-join filter didn't actually exclude regional aggregates, and KPI 1's/
KPI 2-3's reporter side had no filter applied at all) has been fixed: all 4 KPI queries
now filter directly on `area_code`/`reporter_country_code`/`partner_country_code < 5000`.
**Confirmed real counts after re-run:**
- KPI 1: 4,414 rows (pesticide import dependency, `TEST_ITEM = "Pesticides (total)"`) —
  down from 4,934 pre-fix, consistent with aggregate rows now correctly excluded
- KPI 2: 2,713 rows (export concentration, `MATRIX_ITEM = "Food preparations n.e.c."`)
- KPI 3: 2,870 rows (import partner diversification, same `MATRIX_ITEM`)
- KPI 4: 216 countries with both 2001 and 2020 dependency values — down from 242
  pre-fix, consistent with aggregate rows now correctly excluded

**Note on KPI 2's top rows showing 100.0%:** confirmed not a bug — this happens
whenever a country exported `MATRIX_ITEM` to exactly one partner in a given year (real
for small/low-volume exporters like Solomon Islands, Burundi, Bahamas in the sample
output). Sorting descending surfaces these ties at the ceiling first; the full
distribution below 100% is what will be most informative in Power BI, so consider a
histogram or percentile breakdown there rather than only showing the top-N table.

1. ~~Element string casing~~ — **FIXED.**
2. ~~Missing aggregate-exclusion join on `fact_trade__trade`~~ — **FIXED** in KPI 1.
3. ~~KPI 2/3 had no partner-side aggregate-exclusion~~ — **FIXED.** `31_`'s partner-side
   check against the real DB found 9 unmatched partner codes — all uninhabited/remote
   territories (Bouvet Island, Wake Island, etc.), not regional aggregates, so this
   filter has real but small effect.
4. ~~KPI 2's top-partner logic used a fragile `.sort()` + `.first()` pattern~~ —
   **FIXED**, replaced with `.max()` + `.sort_by()` inside the aggregation.
5. ~~No guard when an item has zero/partial coverage~~ — **FIXED**, and this guard is
   exactly what caught the next issue.
6. ~~`TEST_ITEM` was `None`~~ — **FIXED.** Set to `"Pesticides (total)"`.
7. **KPI 1 redefined** to `imports ÷ (imports + exports)` since pesticides have no
   matching `Production` row in `fact_production__crops_livestock`.
8. ~~KPI 2/3 returned 0 rows~~ — **FIXED, root cause confirmed.** `TEST_ITEM` (a
   pesticide) does not exist in `fact_trade__matrix` under any name — confirmed via a
   direct shell query (`LIKE '%pestic%'` etc. → 0 rows). `fact_trade__matrix` has no
   pesticide items at all; its universe is processed food/beverage/agricultural
   commodities. KPI 2/3 now use a separate `MATRIX_ITEM = "Food preparations n.e.c."`
   (595,831 rows, 186 reporters — the best-covered item in that table).

**Known, accepted trade-off:** KPI 1/4 (pesticides) and KPI 2/3 (food preparations) now
describe two different commodities rather than one consistent item across all four
KPIs, unlike Problems 1/2 where all tables shared crop-level naming. This is a genuine
data-scope constraint (`fact_trade__trade` and `fact_trade__matrix` simply don't
overlap on pesticides), not an oversight — flag clearly in any Power BI view that
combines these KPIs.

**Data quality, confirmed from `31_`:** no `year` nulls, no missingness issues in key
columns across any of the 4 source tables. Bilateral join-key type mismatch confirmed
but not relevant to the KPIs above.

**Still open, not blocking a first run:**
- `SIGNIFICANT_PARTNER_SHARE_PCT = 5.0` — revisit once the real partner-share
  distribution is visible.
- Whether `fact_trade__fertilizers_detailedtradematrix` deserves its own KPI.
- Whether the notebook should be split into two clearly-labeled halves (pesticide
  dependency vs. food-preparation trade concentration) rather than presented as one
  unified "Trade Intelligence" KPI set, given they're different commodities.

**Next step:** run this notebook end-to-end against the live DB and record the real row
counts for all 4 KPIs (KPI 1/4 should match the row counts already seen: KPI 1 = 4,934
rows, KPI 4 = 242 countries with both years). Once KPI 2/3's real output is confirmed
non-empty, this notebook reaches the same "locked" status `12_` and `22_` have.

## Close connection

In [11]:
conn.close()
print("Connection closed")

Connection closed
